# Final Project
## Machine Learning for Neuroscience
### Gaia Negev and Tzlil Tabib

In [ ]:
import json
import pandas as pd
import numpy as np
import seaborn as sns
import ast
import shap
import warnings
import joblib
import matplotlib.pyplot as plt
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import auc as sklearn_auc
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.metrics import roc_curve, roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA

# Initial Data Handling
Loading, organizing and dividing to train and test

In [ ]:
# # load data from json
# with open('data/emoset_challenge_1000_augmented.json', 'r', encoding='utf-8') as f:
#     data = json.load(f)

# # data to pd.DataFrame
# df = pd.DataFrame(data)
# df_annotations = pd.DataFrame(df['annotations'].tolist())
# df = pd.concat([df.drop('annotations', axis=1), df_annotations], axis=1)
# df.head()

# # remove image_name column because it's duplicated in image_id
# df = df.drop('image_name', axis=1)
# # make image_id the index
# df = df.set_index('image_id')
# df.head()

# # divide data to train and test
# train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
# print(f'Train size: {len(train_df)}, Test size: {len(test_df)}')

# # save division to csv
# train_df.to_csv('data/train.csv')
# test_df.to_csv('data/test.csv')

In [ ]:
# load the data 
train_df = pd.read_csv('data/train.csv', index_col='image_id')
test_df = pd.read_csv('data/test.csv', index_col='image_id')

# EDA

In [ ]:
train_df.info()

# null analysis
train_df.isnull().sum()

In [ ]:
total = len(train_df)
null_counts = train_df.isnull().sum()
null_pct = (null_counts / total * 100).round(1)

null_summary = pd.DataFrame({
    'null_count': null_counts,
    'null_pct': null_pct
}).sort_values('null_pct', ascending=False)

print(null_summary.to_string())

# Plot
plt.figure(figsize=(8, 5))
colors = ['red' if p > 50 else 'orange' if p > 10 else 'steelblue' 
          for p in null_summary['null_pct']]
plt.barh(null_summary.index, null_summary['null_pct'], color=colors)
plt.axvline(50, color='red', linestyle='--', linewidth=1, alpha=0.5, label='50% threshold')
plt.xlabel('% Missing')
plt.title('Missing values per variable')
plt.legend()
plt.tight_layout()
plt.show()

Since most of the samples are missing facial_expression, human_action and scence - we'll consider leaving these variables out of the analysis. 

In [ ]:
# Emotion valence mapping - color coding in visualizations
# Positive emotions → warm colors
# Negative emotions → cool colors

emotion_colors = {
    'amusement':   "#EE77BC", 
    'excitement':  "#D64772", 
    'contentment': "#F3958E", 
    'awe':         '#880E4F', 
    'sadness':     '#1565C0', 
    'fear':        "#84BEAD",  
    'disgust':     "#617BD1", 
    'anger':       "#1E6391",  
}
emotion_names = list(emotion_colors.keys())
# Convert to list ordered by emotion_names (for indexing by class number)
colors = [emotion_colors[e] for e in emotion_names]

def plot_emotion_legend(ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(3, 1))
    
    positive = ['amusement', 'excitement', 'contentment', 'awe']
    negative = ['sadness', 'fear', 'disgust', 'anger']
    
    for i, e in enumerate(positive):
        ax.scatter([], [], color=emotion_colors[e], label=f'{e} (+)', s=80)
    for i, e in enumerate(negative):
        ax.scatter([], [], color=emotion_colors[e], label=f'{e} (−)', s=80)
    
    ax.legend(ncol=2, fontsize=9, title='Emotion valence')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

plot_emotion_legend()

In [ ]:
# emotion distribution to see if there is a class imbalance
emotion_counts = train_df['emotion'].value_counts()
print(emotion_counts)

emotion_counts_ordered = train_df['emotion'].value_counts().reindex(emotion_names)
plt.figure(figsize=(8, 8))
plt.pie(
    emotion_counts_ordered,
    labels=emotion_counts_ordered.index,
    colors=[emotion_colors[e] for e in emotion_counts_ordered.index],
    autopct='%1.1f%%',
    startangle=90
)
plt.title('Emotion distribution in training set')
plt.axis('equal')
plt.show()

# Data Preprocessing Plan 
1. Numeric variables: brightness, colorfulness; standarize and fill nulls using median
2. Category variables: emotion; label encoding
3. Text variables: description, viewer_feelings, object; tfidf

## Numerics Variables

In [ ]:
train_df.describe()

# Impute the few missing numeric values with median (preprocessing hasn't run yet)
brightness_col = train_df['brightness'].fillna(train_df['brightness'].median())
colorfulness_col = train_df['colorfulness'].fillna(train_df['colorfulness'].median())
X_numeric = np.column_stack([brightness_col, colorfulness_col])

# visualize the distributions of brightness and colorfulness by histogram
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(X_numeric[:, 0], kde=True, color='orange')
plt.title('Brightness Distribution (raw values)')
plt.xlabel('Brightness')
plt.subplot(1, 2, 2)
sns.histplot(X_numeric[:, 1], kde=True, color='blue')
plt.title('Colorfulness Distribution (raw values)')
plt.xlabel('Colorfulness')
plt.tight_layout()
plt.show()

# Standardize inline
scaler_eda = StandardScaler()
X_numeric_scaled = scaler_eda.fit_transform(X_numeric)

print(f"Correlation between brightness and colorfulness: {np.corrcoef(X_numeric_scaled[:, 0], X_numeric_scaled[:, 1])[0, 1]:.2f}")

We see a low correlation between brightness and colorfulness, suggesting each holds unique information.

## Labeling emotion - target variable

In [ ]:
# Encode emotion labels inline
le_eda = LabelEncoder()
target = le_eda.fit_transform(train_df['emotion'])
emotion_names_eda = le_eda.classes_

## Text Variables - TFIDF

max_features of the TF-IDF were chosen on a rule of thumb, max_features = 10% of N samples

In [ ]:
def prefix_text(text, prefix):
    if not isinstance(text, str) or text.strip() == '':
        return ''
    words = [w for w in text.lower().split() if w not in ENGLISH_STOP_WORDS]
    return ' '.join(f"{prefix}_{w}" for w in words)

def prefix_object_list(text, prefix):
    """Handle object column which contains list strings like ['Tree', 'Plant']"""
    if not isinstance(text, str) or text.strip() == '':
        return ''
    try:
        items = ast.literal_eval(text)  # parse ['Tree', 'Plant'] -> ['Tree', 'Plant']
        if isinstance(items, list):
            words = [item.lower().strip() for item in items if item.strip()]
            return ' '.join(f"{prefix}_{w}" for w in words)
    except:
        pass
    # fallback: treat as plain text
    return prefix_text(text, prefix)

train_df['combined_text'] = (
    train_df['description'].apply(lambda x: prefix_text(x, 'desc')) + ' ' +
    train_df['viewer_feelings'].apply(lambda x: prefix_text(x, 'feel')) + ' ' +
    train_df['object'].apply(lambda x: prefix_object_list(x, 'obj'))
)

tfidf_combined = TfidfVectorizer(max_features=80)
combined_matrix = tfidf_combined.fit_transform(train_df['combined_text'])

print(f"Combined TF-IDF matrix shape: {combined_matrix.shape}")
print(f"Sample features: {tfidf_combined.get_feature_names_out()[:20]}")

In [ ]:
# present the first 5 rows of combined_matrix
combined_df = pd.DataFrame(combined_matrix[:5].toarray(), columns=tfidf_combined.get_feature_names_out())
print(combined_df.head())
print(combined_df.columns)

# Modeling 

## 1. Tabular model
Steps: 
1. Choose 2-3 potential models
2. Use cross validation on the training set - use GridSearchCV to do hyperparameter tunning
3. Assess feature importance
4. Evaluate performance of best model on the test set and interpret

In [ ]:
# Variables to include:
# - brightness
# - colorfulness
# - features from the text columns (after TF-IDF - description, viewer_feelings, object)

In [ ]:
warnings.filterwarnings('ignore')

# ── 1. Assemble feature matrix ────────────────────────────────────────────────
X_train = hstack([combined_matrix, csr_matrix(X_numeric_scaled)])
le = le_eda
y_train = target
encoder_order = list(le_eda.classes_)
print(f"Feature matrix shape: {X_train.shape}")

# ── 2. Define models with hyperparameter grids ────────────────────────────────
coarse_grids = {
    'Logistic Regression (L1)': (
        LogisticRegression(penalty='l1', solver='saga', max_iter=1000, random_state=42),
        {'C': [0.01, 0.1, 1, 10]}
    ),
    'Logistic Regression (L2)': (
        LogisticRegression(penalty='l2', solver='lbfgs', max_iter=1000, random_state=42),
        {'C': [0.01, 0.1, 1, 10]}
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=42),
        {'n_estimators': [100, 200], 'max_depth': [None, 10]}
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(random_state=42),
        {'n_estimators': [100, 200], 'max_depth': [3, 5]}
    ),
}

# ── 3. Coarse tuning + CV evaluation ─────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}
models = {}  # store best estimators for later use (ROC curves etc.)

for name, (model, grid) in coarse_grids.items():
    gs = GridSearchCV(model, grid, cv=cv, scoring='roc_auc_ovr', n_jobs=-1)
    gs.fit(X_train, y_train)
    best = gs.best_estimator_

    # Compute all metrics on best estimator
    acc       = cross_val_score(best, X_train, y_train, cv=cv, scoring='accuracy')
    precision = cross_val_score(best, X_train, y_train, cv=cv, scoring='precision_macro')
    recall    = cross_val_score(best, X_train, y_train, cv=cv, scoring='recall_macro')
    auc       = cross_val_score(best, X_train, y_train, cv=cv, scoring='roc_auc_ovr')

    results[name] = {
        'accuracy':    acc,
        'precision':   precision,
        'recall':      recall,
        'auc':         auc,
        'best_params': gs.best_params_,
    }
    models[name] = best  # save for ROC curves

    print(f"\n{name}  best_params={gs.best_params_}")
    for metric, scores in results[name].items():
        if metric == 'best_params':
            continue
        print(f"  {metric:22s} mean={scores.mean():.3f}  std={scores.std():.3f}")

# ── 4. Select best model by AUC ───────────────────────────────────────────────
best_model_name = max(
    {k: v for k, v in results.items()},
    key=lambda x: results[x]['auc'].mean()
)
best_model = models[best_model_name]
print(f"\nBest model: {best_model_name}  AUC={results[best_model_name]['auc'].mean():.3f}")

In [ ]:
# ── 1. ROC curves (models dict now contains tuned estimators) ─────────────────
y_bin = label_binarize(y_train, classes=range(len(encoder_order)))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))  # 4 models → 2x2 grid
axes = axes.flatten()

for ax, (name, model) in zip(axes, models.items()):
    y_prob = cross_val_predict(
        model, X_train, y_train, cv=cv, method='predict_proba'
    )
    for i, emotion in enumerate(encoder_order):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_auc = sklearn_auc(fpr, tpr)
        ax.plot(fpr, tpr, color=emotion_colors[emotion], linewidth=1.5,
                label=f"{emotion} (AUC={roc_auc:.2f})")
    
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.5)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(fontsize=7, loc='lower right')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])

plt.suptitle('ROC Curves per Emotion — All Models (OvR, Cross-Validated)', fontsize=14)
plt.tight_layout()
plt.show()

# ── 3. Score per fold plot ────────────────────────────────────────────────────
plt.figure(figsize=(10, 5))
for name, metrics in results.items():
    plt.plot(range(1, 6), metrics['auc'], marker='o', label=name)  # plot AUC per fold
plt.xlabel('Fold')
plt.xticks(range(1, 6))
plt.ylabel('AUC (OvR)')
plt.title('Model Performance per Fold')
plt.legend()
plt.tight_layout()
plt.show()

The striking thing is that AUC is much higher than accuracy. This tells you:

The models are good at ranking and separating emotions from each other (high AUC)
But struggles to make confident final predictions (low accuracy)

Feature importance analysis on the best model

In [ ]:
# Get feature names (TF-IDF + numeric)
feature_names = list(tfidf_combined.get_feature_names_out()) + ['brightness', 'colorfulness']

# ── For Logistic Regression ──────────────────────────────────────────────────
# coef_ shape is (n_classes, n_features) - one row per emotion
coef_df = pd.DataFrame(
    best_model.coef_,
    index=emotion_names,
    columns=feature_names
)

# Plot top N features per emotion
n_top = 10
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, emotion in enumerate(emotion_names):
    top_features = coef_df.loc[emotion].abs().nlargest(n_top)
    colors = ['#C2447F' if coef_df.loc[emotion, f] < 0 else "#4878BE" for f in top_features.index]
    axes[i].barh(top_features.index, coef_df.loc[emotion, top_features.index], color=colors)
    axes[i].set_title(emotion)
    axes[i].axvline(0, color='black', linewidth=0.5)

plt.suptitle('Top features per emotion (Logistic Regression)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Convert to DataFrame with feature names BEFORE passing to SHAP
X_train_dense = pd.DataFrame(
    X_train.toarray(),
    columns=feature_names
)

explainer = shap.LinearExplainer(best_model, X_train_dense)
shap_values = explainer(X_train_dense)
for i, emotion in enumerate(emotion_names):
    print(f"\n--- {emotion} ---")
    shap.plots.beeswarm(shap_values[:, :, i], max_display=10, show=True)

In [ ]:
import matplotlib.patches as mpatches

X_train_dense = pd.DataFrame(
    X_train.toarray(),
    columns=feature_names
)

explainer = shap.LinearExplainer(best_model, X_train_dense)
shap_values = explainer(X_train_dense)

# ── Plot all emotions in one figure ──────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(24, 12))
axes = axes.flatten()

for i, emotion in enumerate(encoder_order):
    plt.sca(axes[i])  # set current axis for shap to draw on
    
    shap.plots.beeswarm(
        shap_values[:, :, i],
        max_display=10,
        show=False,        # don't render yet
        plot_size=None     # use our axis size
    )
    
    # Add emotion color as a title badge
    color = emotion_colors[emotion]
    axes[i].set_title(emotion, fontsize=12, fontweight='bold',
                      color='white',
                      bbox=dict(facecolor=color, edgecolor=color, 
                                boxstyle='round,pad=0.3'))

plt.suptitle('SHAP Beeswarm — Feature importance per emotion\n(warm = positive, cool = negative)',
             fontsize=14)
plt.tight_layout()
plt.show()

Choosing the most important features using SHAP
- We used cross validation to determine the best N(features) - that resulted in the maximum AUC 

In [ ]:
# ── 1. Get mean absolute SHAP per feature (across all emotions) ──────────────
# shap_values.values shape: (n_samples, n_features, n_classes)
mean_shap_per_feature = np.abs(shap_values.values).mean(axis=(0, 2))  
# average over samples AND classes → shape: (n_features,)

# Create a ranked dataframe
shap_importance_df = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': mean_shap_per_feature
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print(shap_importance_df.head(20))

# Plot
plt.figure(figsize=(10, 6))
plt.barh(shap_importance_df['feature'][:20][::-1], 
         shap_importance_df['mean_abs_shap'][:20][::-1], color='lightpink')
plt.xlabel('Mean absolute SHAP value')
plt.title('Top 20 features by SHAP importance')
plt.tight_layout()
plt.show()

# ── 2. Try different feature cutoffs using CV ────────────────────────────────
# Instead of arbitrarily picking top-N, test several thresholds

cutoffs = [10, 20, 30, 40, 50, 60, 82]  # 82 = all features
cutoff_results = {}

for n in cutoffs:
    # Select top N feature indices
    top_n_features = shap_importance_df['feature'][:n].tolist()
    top_n_idx = [feature_names.index(f) for f in top_n_features]
    
    # Subset the feature matrix
    X_subset = X_train_dense.iloc[:, top_n_idx]
    
    # CV with best model
    scores = cross_val_score(
        LogisticRegression(penalty='l2', solver='lbfgs', max_iter=1000, random_state=42),
        X_subset, y_train, cv=cv, scoring='roc_auc_ovr'
    )
    cutoff_results[n] = scores
    print(f"Top {n:3d} features — AUC mean={scores.mean():.3f}  std={scores.std():.3f}")

# ── 3. Plot performance vs number of features ────────────────────────────────
means = [cutoff_results[n].mean() for n in cutoffs]
stds = [cutoff_results[n].std() for n in cutoffs]

plt.figure(figsize=(8, 4))
plt.plot(cutoffs, means, marker='o', color='hotpink', linewidth=2)
plt.fill_between(cutoffs, 
                 [m - s for m, s in zip(means, stds)],
                 [m + s for m, s in zip(means, stds)],
                 alpha=0.2, color='lightpink')
plt.axhline(means[-1], color='gray', linestyle='--', linewidth=1, label='All features')
plt.xlabel('Number of features (top N by SHAP)')
plt.ylabel('CV AUC (OvR)')
plt.title('Model performance vs feature count')
plt.legend()
plt.tight_layout()
plt.show()

# ── 4. Select best cutoff and retrain final model ────────────────────────────
best_n = cutoffs[np.argmax(means)]
print(f"Best cutoff: top {best_n} features")

top_features_final = shap_importance_df['feature'][:best_n].tolist()
top_idx_final = [feature_names.index(f) for f in top_features_final]

X_train_final = X_train_dense.iloc[:, top_idx_final]

# Retrain with GridSearchCV on reduced feature set
grid_search_final = GridSearchCV(
    LogisticRegression(penalty='l2', solver='lbfgs', max_iter=1000, random_state=42),
    {'C': [0.001, 0.01, 0.1, 1, 10, 100]},
    cv=cv, scoring='roc_auc_ovr', verbose=1
)
grid_search_final.fit(X_train_final, y_train)
best_model_final = grid_search_final.best_estimator_

print(f"Best C: {grid_search_final.best_params_}")
print(f"Best CV AUC: {grid_search_final.best_score_:.3f}")

In [ ]:
# save the best model with the best_n features for later evaluation on the test set - save best_shap_features_model.pkl
# ── 1. Save the fitted objects ────────────────────────────────────────────────
joblib.dump(best_model, 'data/best_model.pkl')
# Save the top 60 feature names from SHAP
top_60_features = shap_importance_df['feature'][:60].tolist()
joblib.dump(top_60_features, 'data/top_60_features.pkl')

print("Saved:")
print(top_60_features)

top_60_idx = [feature_names.index(f) for f in top_60_features]

# Also subset train to top 60 for consistency
X_train_60 = X_train_dense.iloc[:, top_60_idx]

# Refit best model on full train with top 60 features
best_model.fit(X_train_60, y_train)

Evaluate on test set:

Apply same preprocessing to the test set variables

In [ ]:
# Apply the SAME preprocessing to test_df
text_columns = ['description', 'viewer_feelings', 'object']
test_df[text_columns] = test_df[text_columns].fillna('')
test_df['combined_text'] = (
    test_df['description'].apply(lambda x: prefix_text(x, 'desc')) + ' ' +
    test_df['viewer_feelings'].apply(lambda x: prefix_text(x, 'feel')) + ' ' +
    test_df['object'].apply(lambda x: prefix_object_list(x, 'obj'))
)

# Use transform on test data
test_tfidf = tfidf_combined.transform(test_df['combined_text'])
test_numeric = scaler_eda.transform(test_df[['brightness', 'colorfulness']].fillna(
    train_df[['brightness', 'colorfulness']].median()
))

y_test = le_eda.transform(test_df['emotion'])

Applying the model on the test set and visualizing the confusion matrix

In [ ]:

# Combine into full feature matrix
X_test_full = hstack([test_tfidf, csr_matrix(test_numeric)])

# Convert to dense and select only top 60 features
X_test_full_dense = pd.DataFrame(
    X_test_full.toarray(),
    columns=feature_names
)
X_test_60 = X_test_full_dense.iloc[:, top_60_idx]
y_pred = best_model.predict(X_test_60)
y_prob_test = best_model.predict_proba(X_test_60)

print(classification_report(y_test, y_pred, target_names=emotion_names))

# AUC on test set
y_bin_test = label_binarize(y_test, classes=range(len(emotion_names)))
test_auc = roc_auc_score(y_bin_test, y_prob_test, multi_class='ovr', average='macro')
print(f"Test AUC (macro OvR): {test_auc:.3f}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=encoder_order)
fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(ax=ax, xticks_rotation=45, cmap='RdPu')
plt.title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.show()

Comparing predictions of negative vs. positive emotions

In [ ]:
# ── 1. Map emotions to valence ────────────────────────────────────────────────
positive_emotions = ['amusement', 'excitement', 'contentment', 'awe']
negative_emotions = ['anger', 'disgust', 'fear', 'sadness']

valence_map = {e: 'positive' for e in positive_emotions}
valence_map.update({e: 'negative' for e in negative_emotions})

# Map predictions and true labels to valence
y_test_valence = [valence_map[encoder_order[i]] for i in y_test]
y_pred_valence = [valence_map[encoder_order[i]] for i in y_pred]

# ── 2. Confusion matrix ───────────────────────────────────────────────────────

cm_valence = confusion_matrix(y_test_valence, y_pred_valence, labels=['positive', 'negative'])

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_valence, display_labels=['positive', 'negative'])
disp.plot(ax=ax, colorbar=False, cmap='RdPu', xticks_rotation=0)

# Color the tick labels by valence
ax.get_xticklabels()[0].set_color("#C2447F")
ax.get_xticklabels()[1].set_color("#106799")
ax.get_yticklabels()[0].set_color('#C2447F')
ax.get_yticklabels()[1].set_color('#106799')

plt.title('Valence confusion matrix\n(positive vs negative emotions)', fontsize=12)
plt.tight_layout()
plt.show()

# ── 3. Print valence-level accuracy ──────────────────────────────────────────
print(classification_report(y_test_valence, y_pred_valence, 
                            target_names=['positive', 'negative']))

## 2. Embedding model

### Applying PCA on the embedded variables

In [ ]:
# Parse embedding strings to numpy arrays
train_df['embedding_parsed'] = train_df['embedding'].apply(ast.literal_eval)
train_df['desc_emb_parsed'] = train_df['description_embedding'].apply(ast.literal_eval)
train_df['feel_emb_parsed'] = train_df['viewer_feelings_embedding'].apply(ast.literal_eval)

# Stack into matrices
E_image = np.stack(train_df['embedding_parsed'].values)         # image CLIP embedding
E_desc  = np.stack(train_df['desc_emb_parsed'].values)          # description text embedding
E_feel  = np.stack(train_df['feel_emb_parsed'].values)          # viewer feelings text embedding

print(f"Image embedding shape:       {E_image.shape}")
print(f"Description embedding shape: {E_desc.shape}")
print(f"Feelings embedding shape:    {E_feel.shape}")
# Expect something like (800, 512) or (800, 1536) depending on the model used

In [ ]:
# ── 1. Standardize each embedding separately ─────────────────────────────────
# Important before PCA - embeddings may have different scales
scaler_image = StandardScaler()
scaler_desc  = StandardScaler()
scaler_feel  = StandardScaler()

E_image_scaled = scaler_image.fit_transform(E_image)
E_desc_scaled  = scaler_desc.fit_transform(E_desc)
E_feel_scaled  = scaler_feel.fit_transform(E_feel)

# ── 2. PCA on each embedding separately ──────────────────────────────────────
# Keep enough components to explain ~90% variance
def fit_pca_with_plot(X, name, max_components=500):
    pca = PCA(n_components=max_components)
    pca.fit(X)
    
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    n_90 = np.argmax(cumvar >= 0.90) + 1
    
    plt.figure(figsize=(7, 4))
    plt.plot(cumvar, linewidth=2)
    plt.axhline(0.90, color='red', linestyle='--', label='90% variance')
    plt.axvline(n_90, color='orange', linestyle='--', label=f'{n_90} components')
    plt.xlabel('Number of components')
    plt.ylabel('Cumulative explained variance')
    plt.title(f'PCA — {name}')
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    n_90 = np.argmax(cumvar >= 0.90) + 1
    if n_90 == 1 and cumvar[0] < 0.90:  # threshold never reached
        n_90 = max_components  # just use all 100
    print(f"{name}: 90% variance not reached in {max_components} components, using all {max_components}")
    return pca, n_90

pca_image, n_image = fit_pca_with_plot(E_image_scaled, 'Image embedding')
pca_desc,  n_desc  = fit_pca_with_plot(E_desc_scaled,  'Description embedding')
pca_feel,  n_feel  = fit_pca_with_plot(E_feel_scaled,  'Feelings embedding')

Estimate four types of models on the n(features) from PCA - and tune hyperparameters all models

In [ ]:

X_emb = np.hstack([
    pca_image.transform(E_image_scaled)[:, :n_image],
    pca_desc.transform(E_desc_scaled)[:, :n_desc],
    pca_feel.transform(E_feel_scaled)[:, :n_feel]
])

coarse_grids = {
    'Logistic Regression (L2)': (
        LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42),
        {'C': [0.01, 0.1, 1, 10]}
    ),
    'Logistic Regression (L1)': (
        LogisticRegression(penalty='l1', solver='saga', max_iter=1000, random_state=42),
        {'C': [0.01, 0.1, 1, 10]}
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=42),
        {'n_estimators': [100, 200], 'max_depth': [None, 10]}
    ),
    'KNN': (
        KNeighborsClassifier(),
        {'n_neighbors': list(range(1, 20, 2))}
    ),
}

emb_results = {}
for name, (model, grid) in coarse_grids.items():
    gs = GridSearchCV(model, grid, cv=cv, scoring='roc_auc_ovr', n_jobs=-1)
    gs.fit(X_emb, y_train)
    
    # Get CV accuracy for the best params
    acc = cross_val_score(gs.best_estimator_, X_emb, y_train, cv=cv, scoring='accuracy').mean()
    
    emb_results[name] = {
        'accuracy':       acc,
        'auc':            gs.best_score_,
        'best_params':    gs.best_params_,
        'best_estimator': gs.best_estimator_
    }
    print(f"{name:30s}  acc={acc:.3f}  auc={gs.best_score_:.3f}  params={gs.best_params_}")

# ── Select best model by AUC ──────────────────────────────────────────────────
best_emb_name = max(emb_results, key=lambda x: emb_results[x]['auc'])
best_emb_model = emb_results[best_emb_name]['best_estimator']
print(f"\nBest model: {best_emb_name}  AUC={emb_results[best_emb_name]['auc']:.3f}")

In [ ]:
# TODO: rationalize choosing the best model based on AUC

Visualize model comparison

In [ ]:
# ── Plot 1: Bar chart with error bars ─────────────────────────────────────────
names  = list(emb_results.keys())
aucs   = [emb_results[n]['auc'] for n in names]
accs   = [emb_results[n]['accuracy'] for n in names]
colors = ['coral' if n == best_emb_name else 'steelblue' for n in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, aucs, width, label='AUC',      color=colors, alpha=0.9)
bars2 = ax.bar(x + width/2, accs, width, label='Accuracy', color=colors, alpha=0.5)

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.set_title('Embedding model comparison — AUC and Accuracy\n(coral = best model)')
ax.legend()
ax.axhline(1/8, color='gray', linestyle='--', linewidth=1, label='Random chance')

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

# ── Plot 2: ROC curves ────────────────────────────────────────────────────────
y_bin = label_binarize(y_train, classes=range(len(encoder_order)))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (name, metrics) in zip(axes, emb_results.items()):
    y_prob = cross_val_predict(
        metrics['best_estimator'], X_emb, y_train, cv=cv, method='predict_proba'
    )
    for i, emotion in enumerate(encoder_order):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_auc = sklearn_auc(fpr, tpr)
        ax.plot(fpr, tpr, color=emotion_colors[emotion], linewidth=1.5,
                label=f"{emotion} (AUC={roc_auc:.2f})")

    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.5)
    ax.set_title(f"{name}\nbest params: {metrics['best_params']}", fontsize=10)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(fontsize=7, loc='lower right')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])

plt.suptitle('ROC Curves — Embedding Models (OvR, Cross-Validated)', fontsize=13)
plt.tight_layout()
plt.show()

The model is better at ranking and separating emotions from each other (high AUC)
than at making final predictions (lower accuracy, but still good)
This typically happens when classes overlap in the embedding space — the model knows "this image is more fear-like than amusement-like" but isn't confident enough to commit to a single label

Choosing the number of PCs using CV

In [ ]:
# ── CV-based PCA component selection ─────────────────────────────────────────
n_components_to_try = range(10, 230, 10)
cv_components = {}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (E_scaled, name) in zip(axes, [
    (E_image_scaled, 'Image embedding'),
    (E_desc_scaled,  'Description embedding'),
    (E_feel_scaled,  'Feelings embedding')
]):
    auc_scores = []
    
    for n in n_components_to_try:
        pca_temp = PCA(n_components=n, random_state=42).fit(E_scaled)
        X_temp = pca_temp.transform(E_scaled)
        
        score = cross_val_score(
            LogisticRegression(C=0.01, solver='lbfgs', max_iter=1000, random_state=42),
            X_temp, y_train, cv=cv, scoring='roc_auc_ovr'
        ).mean()
        auc_scores.append(score)
    
    # Best n for this embedding
    best_n = n_components_to_try[np.argmax(auc_scores)]
    cv_components[name] = best_n
    
    # Plot
    ax.plot(n_components_to_try, auc_scores, marker='o', linewidth=2, color='steelblue')
    ax.axvline(best_n, color='red', linestyle='--', linewidth=1.5,
               label=f'Best: {best_n} components (AUC={max(auc_scores):.3f})')
    ax.scatter([best_n], [max(auc_scores)], color='red', s=80, zorder=5)
    ax.set_xlabel('Number of components')
    ax.set_ylabel('CV AUC (OvR)')
    ax.set_title(name)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('CV-based PCA component selection (LR L2, C=0.01)', fontsize=13)
plt.tight_layout()
plt.show()

n_image_final = cv_components['Image embedding']
n_desc_final  = cv_components['Description embedding']
n_feel_final  = cv_components['Feelings embedding']

print(f"Components selected — image: {n_image_final}, description: {n_desc_final}, feelings: {n_feel_final}")

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

pca_2d = PCA(n_components=2, random_state=42)
X_2d_pca = pca_2d.fit_transform(X_emb)

tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_2d_tsne = tsne.fit_transform(X_emb)

for ax, X_2d, title in zip(axes, [X_2d_pca, X_2d_tsne], ['PCA', 't-SNE']):
    for i, emotion in enumerate(encoder_order):
        mask = y_train == i
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   label=emotion,
                   color=emotion_colors[emotion],
                   alpha=0.6, s=30, edgecolors='none')
    ax.set_title(f'Embedding space — {title}', fontsize=12)
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')
    ax.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.suptitle('Emotion clusters in embedding space', fontsize=14)
plt.tight_layout()
plt.show()

Apply preproccessing of embeddings on the test set

In [ ]:
# ── Preprocess test embeddings ────────────────────────────────────────────────
test_df['embedding_parsed'] = test_df['embedding'].apply(ast.literal_eval)
test_df['desc_emb_parsed']  = test_df['description_embedding'].apply(ast.literal_eval)
test_df['feel_emb_parsed']  = test_df['viewer_feelings_embedding'].apply(ast.literal_eval)

E_image_test = np.stack(test_df['embedding_parsed'].values)
E_desc_test  = np.stack(test_df['desc_emb_parsed'].values)
E_feel_test  = np.stack(test_df['feel_emb_parsed'].values)

# Scale using train scalers (transform only, never fit)
E_image_test_scaled = scaler_image.transform(E_image_test)
E_desc_test_scaled  = scaler_desc.transform(E_desc_test)
E_feel_test_scaled  = scaler_feel.transform(E_feel_test)

In [ ]:
# ── Rebuild X_emb with CV-selected components (same as test) ─────────────────
print(f"CV-selected components: image={n_image_final}, desc={n_desc_final}, feel={n_feel_final}")

X_emb_final = np.hstack([
    pca_image.transform(E_image_scaled)[:, :n_image_final],
    pca_desc.transform(E_desc_scaled)[:, :n_desc_final],
    pca_feel.transform(E_feel_scaled)[:, :n_feel_final]
])

X_emb_test = np.hstack([
    pca_image.transform(E_image_test_scaled)[:, :n_image_final],
    pca_desc.transform(E_desc_test_scaled)[:, :n_desc_final],
    pca_feel.transform(E_feel_test_scaled)[:, :n_feel_final]
])

print(f"Train: {X_emb_final.shape}")
print(f"Test:  {X_emb_test.shape}")  # must match on axis 1
assert X_emb_final.shape[1] == X_emb_test.shape[1], "Shape mismatch!"

# ── Refit and evaluate ────────────────────────────────────────────────────────
best_emb_model = emb_results[best_emb_name]['best_estimator']
best_emb_model.fit(X_emb_final, y_train)

y_pred_emb = best_emb_model.predict(X_emb_test)
y_prob_emb = best_emb_model.predict_proba(X_emb_test)
y_test     = le_eda.transform(test_df['emotion'])

accuracy  = accuracy_score(y_test, y_pred_emb)
precision = precision_score(y_test, y_pred_emb, average='macro', zero_division=0)
recall    = recall_score(y_test, y_pred_emb, average='macro', zero_division=0)

y_bin_test = label_binarize(y_test, classes=range(len(encoder_order)))
test_auc   = roc_auc_score(y_bin_test, y_prob_emb, multi_class='ovr', average='macro')

print(f"\nEmbedding model — Test Set Results ({best_emb_name})")
print(f"  Accuracy:          {accuracy:.3f}")
print(f"  Precision (macro): {precision:.3f}")
print(f"  Recall (macro):    {recall:.3f}")
print(f"  AUC (macro OvR):   {test_auc:.3f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred_emb)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=encoder_order)
fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(ax=ax, xticks_rotation=45, colorbar=True, cmap='RdPu')
plt.title(f'Confusion Matrix — Embedding Model Test Set ({best_emb_name})')
plt.tight_layout()
plt.show()

In [ ]:
# Map predictions and true labels to valence
y_test_emb_valence = [valence_map[encoder_order[i]] for i in y_test]
y_pred_emb_valence = [valence_map[encoder_order[i]] for i in y_pred_emb]  # y_pred_emb not y_pred

cm_emb_valence = confusion_matrix(y_test_emb_valence, y_pred_emb_valence, 
                                   labels=['positive', 'negative'])

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_emb_valence, 
                               display_labels=['Positive', 'Negative'])
disp.plot(ax=ax, colorbar=False, cmap='RdPu', xticks_rotation=0)

ax.get_xticklabels()[0].set_color("#C2447F")
ax.get_xticklabels()[1].set_color("#106799")
ax.get_yticklabels()[0].set_color('#C2447F')
ax.get_yticklabels()[1].set_color('#106799')

plt.title('Valence confusion matrix — Embedding model\n(positive vs negative emotions)', fontsize=12)
plt.tight_layout()
plt.show()

print(classification_report(y_test_emb_valence, y_pred_emb_valence,
                            target_names=['positive', 'negative']))